# IMDN x2 Colab Pilot

This is the first real Phase 3 inference test. It downloads the authors' official IMDN x2 checkpoint, verifies its exact SHA-256 checksum, reconstructs one Set5 image on the GPU, calculates the project's fixed metrics, and saves the result to Google Drive. It does not train a model or run the full dataset.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/divinesta/SuperResolution-Comparative-Analysis.git'
REPO_ROOT = Path('/content/SuperResolution-Comparative-Analysis')
if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')],
    check=True,
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. Select a GPU runtime and reconnect.')
DEVICE = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

In [ ]:
from app.deep_learning.checkpoints import (
    IMDN_CHECKPOINTS,
    download_official_imdn_checkpoint,
)

DATA_ROOT = Path('/content/drive/MyDrive/FYP_SR_Data')
CHECKPOINT_ROOT = DATA_ROOT / 'checkpoints'
provenance = IMDN_CHECKPOINTS[2]
checkpoint_path = download_official_imdn_checkpoint(CHECKPOINT_ROOT, scale=2)
print('Verified checkpoint:', checkpoint_path)
print('SHA-256:', provenance.sha256)
print('Source:', provenance.source_repository)

In [ ]:
from app.config import dataset_hr_directory, dataset_lr_directory
from app.evaluation.images import load_rgb_image, pair_image_paths

hr_directory = dataset_hr_directory('Set5', DATA_ROOT)
lr_directory = dataset_lr_directory('Set5', 2, DATA_ROOT)
pairs = pair_image_paths(hr_directory, lr_directory)
exact_pairs = []
for hr_path, lr_path in pairs:
    hr_image = load_rgb_image(hr_path)
    lr_image = load_rgb_image(lr_path)
    if hr_image.size == (lr_image.width * 2, lr_image.height * 2):
        exact_pairs.append((hr_path, lr_path))
if not exact_pairs:
    raise RuntimeError('Set5 contains no exact native x2 pair for the pilot.')

hr_path, lr_path = exact_pairs[0]
reference_hr = load_rgb_image(hr_path)
lr_image = load_rgb_image(lr_path)
print('Pilot image:', hr_path.name)
print('LR size:', lr_image.size, 'HR size:', reference_hr.size)

In [ ]:
from statistics import mean, median

from app.deep_learning.imdn import (
    load_pretrained_imdn,
    pil_to_tensor,
    tensor_to_pil,
)

model = load_pretrained_imdn(checkpoint_path, scale=2, device=DEVICE)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
input_tensor = pil_to_tensor(lr_image, DEVICE)

with torch.inference_mode():
    for _ in range(3):
        model(input_tensor)
torch.cuda.synchronize()

latencies_ms = []
with torch.inference_mode():
    for _ in range(10):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        output_tensor = model(input_tensor)
        end.record()
        torch.cuda.synchronize()
        latencies_ms.append(start.elapsed_time(end))

reconstruction = tensor_to_pil(output_tensor)
if reconstruction.size != reference_hr.size:
    raise RuntimeError(
        f'Native IMDN output {reconstruction.size} does not match HR {reference_hr.size}.'
    )
print('Parameters:', f'{parameter_count:,}')
print('Mean GPU latency (ms):', round(mean(latencies_ms), 3))
print('Median GPU latency (ms):', round(median(latencies_ms), 3))

In [ ]:
import json
from dataclasses import asdict
from datetime import UTC, datetime

from app.evaluation.metrics import calculate_quality_metrics

metrics = calculate_quality_metrics(reference_hr, reconstruction, border=2)
output_directory = DATA_ROOT / 'results' / 'phase3' / 'pilot' / 'imdn_x2'
output_directory.mkdir(parents=True, exist_ok=True)
image_output_path = output_directory / f'{hr_path.stem}_imdn_x2.png'
reconstruction.save(image_output_path)
record = {
    'dataset': 'Set5',
    'image': hr_path.name,
    'scale': 'x2',
    'method': 'imdn',
    'device': torch.cuda.get_device_name(0),
    'parameter_count': parameter_count,
    'latency_mean_ms': mean(latencies_ms),
    'latency_median_ms': median(latencies_ms),
    **metrics,
    'checkpoint': asdict(provenance),
    'generated_at_utc': datetime.now(UTC).isoformat(),
}
record_output_path = output_directory / f'{hr_path.stem}_imdn_x2_pilot.json'
record_output_path.write_text(json.dumps(record, indent=2) + '\n', encoding='utf-8')
print(json.dumps(metrics, indent=2))
print('Saved image:', image_output_path)
print('Saved pilot record:', record_output_path)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for axis, image, title in zip(
    axes,
    (lr_image, reconstruction, reference_hr),
    ('Prepared LR', 'IMDN x2', 'Reference HR'),
):
    axis.imshow(image)
    axis.set_title(title)
    axis.axis('off')
plt.tight_layout()
plt.show()

## Completion test

The pilot passes when the checkpoint is verified, the model reports approximately 700k parameters, the IMDN output exactly matches the HR dimensions, four metrics print, and both output paths are saved. Send those outputs before moving to x3/x4 or a full-dataset run.